In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\deforestation_detection\\research\\inference_trials'

In [3]:
os.chdir('../../')

In [4]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\deforestation_detection'

In [5]:
import os
import torch
import numpy as np
import segmentation_models_pytorch as smp
from pathlib import Path
import matplotlib.pyplot as plt



Loading the existing model

In [6]:
# 1. Setup device and model configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_path = Path("artifacts/model_trainer/best_unet_model.pth")

# Initialize U-Net architecture matching the training setup
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=4,
    classes=1
).to(device)

# Load trained weights
if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Successfully loaded model weights from {model_path}")
else:
    raise FileNotFoundError(f"Trained model not found at {model_path}! Run training pipeline first.")

model.eval()


Using device: cuda
Successfully loaded model weights from artifacts\model_trainer\best_unet_model.pth


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

 Predictor function: (HELPER FUNCTION) Takes a 4-channel image tensor [C, H, W] or batch [B, C, H, W],
    runs U-Net forward pass with sigmoid, and binarizes at 0.5.

In [7]:
# 2. Prediction Helper Function
def predict_mask(image_tensor):
   
    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0) # Add batch dimension
        
    image_tensor = image_tensor.to(device).float()
    
    with torch.no_grad():
        output = model(image_tensor)
        pred_sigmoid = torch.sigmoid(output)
        pred_mask = (pred_sigmoid > 0.5).float()
        
    return pred_mask.squeeze(0).cpu() # Return [1, H, W] on CPU

Change detecttion function: Compares forest mask at Date 1 (t1) and Date 2 (t2).
    Deforestation = Forest in t1 (1) AND Non-Forest in t2 (0).
    Filters out noise patches smaller than min_pixels.

In [8]:
# 3. Change Detection Function (Step 8)
def detect_deforestation(mask_t1, mask_t2, min_pixels=50):
    
    # t1: 1 (forest), t2: 0 (cleared/non-forest)
    # Cleared mask where t1 is forest and t2 is NOT forest
    cleared_mask = (mask_t1 == 1) & (mask_t2 == 0)
    
    # Optional: Noise filtering can be integrated via connected components if needed,
    # or basic thresholding. Here we verify the logical matrix shape.
    print(f"Detected cleared pixels before filtering: {cleared_mask.sum().item()}")
    
    return cleared_mask.float()

print("Inference and Change Detection helper functions ready!")

Inference and Change Detection helper functions ready!


Testing

In [9]:
# 4. Test Change Detection on actual dataset arrays
import torch

dataset_dict = torch.load("artifacts/data_ingestion/dataset_arrays.pt")
X_full = dataset_dict["X_full"]
y_full = dataset_dict["y_full"]

# Let's simulate Date 1 and Date 2 using two different samples (or sequential patches)
# Sample A as Date 1, Sample B as Date 2
sample_idx_1 = 0
sample_idx_2 = 1

img_t1 = X_full[sample_idx_1]
mask_t1 = y_full[sample_idx_1] # Ground truth or model prediction at t1

img_t2 = X_full[sample_idx_2]
mask_t2 = y_full[sample_idx_2] # Ground truth or model prediction at t2

# Alternatively, run model prediction on both images:
pred_mask_t1 = predict_mask(img_t1)
pred_mask_t2 = predict_mask(img_t2)

# Run change detection
cleared_change_mask = detect_deforestation(pred_mask_t1, pred_mask_t2, min_pixels=50)
print("Change detection test executed successfully!")

Detected cleared pixels before filtering: 0
Change detection test executed successfully!



    Filters out cleared forest patches smaller than min_pixels (~0.5 hectares at 10m resolution).
    cleared_mask: PyTorch tensor or numpy array of shape [H, W] with 1 for cleared, 0 otherwise.

In [10]:
from scipy import ndimage
import numpy as np

def filter_small_cleared_patches(cleared_mask, min_pixels=50):
    
    if isinstance(cleared_mask, torch.Tensor):
        mask_np = cleared_mask.detach().cpu().numpy().astype(int)
    else:
    # Handle batch or channel dims if present
        mask_np = np.squeeze(cleared_mask).astype(int)
        
    # Label connected components of cleared areas
    labeled_array, num_features = ndimage.label(mask_np)
    
    filtered_mask = np.zeros_like(mask_np)
    
    removed_count = 0
    retained_count = 0
    
    for i in range(1, num_features + 1):
        component_mask = (labeled_array == i)
        component_size = component_mask.sum()
        
        if component_size >= min_pixels:
            filtered_mask[component_mask] = 1
            retained_count += 1
        else:
            removed_count += 1
            
    print(f"Noise Filtering: Removed {removed_count} small patches (< {min_pixels} pixels), Retained {retained_count} valid cleared areas.")
    
    return torch.tensor(filtered_mask, dtype=torch.float32)

# Test the filter on our change mask
filtered_change = filter_small_cleared_patches(cleared_change_mask, min_pixels=50)

Noise Filtering: Removed 0 small patches (< 50 pixels), Retained 0 valid cleared areas.
